# Wildlife Re-ID — Final Submission (best model)

Inference-only notebook. It reproduces the best configuration found in `reid_submission_v2.ipynb` (pooled 5-fold OOF MAP@5 = **0.8738**) and writes `submission.csv`.

**Pipeline (per species, species are never mixed)**
1. MiewID descriptors → L2-norm → center → PCA(512, whiten) → L2-norm (fit on train+test of that species)
2. Transductive k-reciprocal re-ranking (k1=20, k2=6, λ=0.3), keep top-200 gallery candidates per query
3. Top-100 candidates per query verified with SIFT: ratio test (0.78) + mutual NN + RANSAC partial-affine (5 px)
4. Blend: `0.2812·S_rerank + 0.7188·S_cos + 0.1906·log(1+max(inliers−3,0)) + viewpoint bonus`
5. Image scores → identity scores with `top2` aggregation (mean of an identity's best 2 images)
6. `new_individual` inserted at a rank decided by a per-species threshold τ (hyena 0.2100, leopard 0.3439)

Only the provided features are used, with classical methods (PCA, kNN-style retrieval, re-ranking, SIFT matching, RANSAC), which is within the competition rules.

**Speed:** if `cache/sift_test_top100.parquet` and `cache/sift_meta.json` from the earlier notebook sit in the working directory, the slow SIFT step is skipped. Otherwise it is recomputed (multi-threaded; OpenCV releases the GIL so this works on Windows/Jupyter too).

## 0. Config & best values

In [ ]:
import os, json, time, glob, warnings, threading
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import cv2
from scipy import sparse
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ---------- data location ----------
def _resolve_data_dir():
    cands = [os.environ.get("DATA_DIR"),
             r"C:\Users\Mayank\Downloads\projectwild3\reid_student_data_main",
             r"C:\Users\Mayank\Downloads\projectwild3",
             "reid_student_data_main", "."]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "train.csv")):
            return c
    raise FileNotFoundError("train.csv not found - set the DATA_DIR environment variable to the folder that contains it.")

DATA_DIR   = _resolve_data_dir()
PARENT_DIR = os.path.dirname(os.path.abspath(DATA_DIR))
CACHE_DIR  = "cache"
OUT_PATH   = "submission.csv"

NEW, SPECIES = "new_individual", ["hyena", "leopard"]

# ---------- best values (from the CV tuning in reid_submission_v2.ipynb) ----------
EMB_CFG = dict(center=True, pca=512)                    # Sub #1 embedding config (agg=max, no DBA/QE, vp=0 -> replaced by re-ranking)
RERANK_K1, RERANK_K2, RERANK_LAM = 20, 6, 0.3           # k-reciprocal re-ranking
CAND_K, TOP_M_SIFT = 200, 100                           # candidates kept / candidates verified with SIFT
SIFT_RATIO, RANSAC_THRESH, RANSAC_ITERS = 0.78, 5.0, 2000

BEST_PARAMS = dict(w_rerank=0.2812, w_sift=0.1906, b_same=-0.0062, b_adj=0.0094, b_opp=0.0125, agg="top2", floor=3)
BEST_TAUS   = dict(hyena=0.2100, leopard=0.3439)

# If best_config_v2.json from the tuning run is present, use its full-precision values (rounded ones above are the fallback).
_cfgp = next((p for p in [os.path.join(d, "best_config_v2.json") for d in [".", DATA_DIR, PARENT_DIR]] if os.path.isfile(p)), None)
if _cfgp:
    _j = json.load(open(_cfgp))
    BEST_PARAMS, BEST_TAUS = _j["params"], _j["tau"]
    print("Loaded full-precision params/taus from", _cfgp)
else:
    print("best_config_v2.json not found - using the hard-coded best values.")
print("params:", BEST_PARAMS)
print("taus  :", BEST_TAUS)
assert CAND_K >= TOP_M_SIFT
print("DATA_DIR:", DATA_DIR)

## 1. Load data

In [ ]:
def load_data(d=DATA_DIR):
    tr = pd.read_csv(f"{d}/train.csv")
    te = pd.read_csv(f"{d}/test.csv")
    ids = pd.read_csv(f"{d}/miewid_ids.csv")
    col = "id" if "id" in ids.columns else ids.columns[0]
    F = np.load(f"{d}/miewid_features.npy").astype(np.float32)
    pos = pd.Series(np.arange(len(ids)), index=ids[col].values)
    Xtr, Xte = F[pos.loc[tr.id].values], F[pos.loc[te.id].values]      # KeyError if ids misalign
    tr["viewpoint"] = tr["viewpoint"].fillna("unk")
    te["viewpoint"] = te["viewpoint"].fillna("unk")
    return tr, te, Xtr, Xte

tr, te, Xtr, Xte = load_data()
print(tr.shape, te.shape, Xtr.shape, Xte.shape)

## 2. Embeddings (per species)

In [ ]:
def l2n(X):
    return X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-12)

def species_embeddings(Xtr_sp, Xte_sp, cfg):
    X = l2n(np.vstack([Xtr_sp, Xte_sp]).astype(np.float32))
    if cfg["center"]:
        X = l2n(X - X.mean(0, keepdims=True))
    if cfg["pca"]:
        X = l2n(PCA(cfg["pca"], whiten=True, random_state=0).fit_transform(X).astype(np.float32))
    return X[:len(Xtr_sp)], X[len(Xtr_sp):]

code_tr = np.full(len(tr), -1, np.int32)          # identity code (index into CLASSES[sp]) per train row
CLASSES, SUB, TE_SUB, EMB = {}, {}, {}, {}
for sp in SPECIES:
    m_tr, m_te = (tr.species == sp).values, (te.species == sp).values
    SUB[sp], TE_SUB[sp] = np.flatnonzero(m_tr), np.flatnonzero(m_te)
    EMB[sp] = species_embeddings(Xtr[m_tr], Xte[m_te], EMB_CFG)
    CLASSES[sp], inv = np.unique(tr.identity.values[m_tr], return_inverse=True)
    code_tr[SUB[sp]] = inv
    print(f"{sp}: train={m_tr.sum()} test={m_te.sum()} identities={len(CLASSES[sp])}")

## 3. k-reciprocal re-ranking (Zhong et al., 2017)
Queries and gallery are stacked, so query-to-query affinities help when the same new individual appears in several test images.

In [ ]:
def k_reciprocal_rerank(Q, G, k1=RERANK_K1, k2=RERANK_K2, lam=RERANK_LAM, chunk=1024):
    """Q (nq,d), G (ng,d): L2-normalised. Returns (S_rerank (nq,ng), S_cos (nq,ng))."""
    Q = np.ascontiguousarray(Q, dtype=np.float32); G = np.ascontiguousarray(G, dtype=np.float32)
    nq, ng = len(Q), len(G)
    X = np.vstack([Q, G]); N = nq + ng
    S_cos = Q @ G.T

    kk = min(k1, N - 1)
    D = np.empty((N, N), np.float32)                      # row-max-normalised squared euclidean distance
    nn = np.empty((N, kk + 1), np.int32)
    for a in range(0, N, chunk):
        b = min(a + chunk, N)
        d = 2.0 - 2.0 * (X[a:b] @ X.T)
        np.maximum(d, 0, out=d)
        d /= np.maximum(d.max(1, keepdims=True), 1e-12)
        D[a:b] = d
        part = np.argpartition(d, kk, axis=1)[:, :kk + 1]
        o = np.argsort(np.take_along_axis(d, part, 1), axis=1)
        nn[a:b] = np.take_along_axis(part, o, 1)
    ar = np.arange(N)
    for i in np.flatnonzero(nn[:, 0] != ar):             # exact duplicates can displace self from position 0
        w = np.flatnonzero(nn[i] == i)
        if len(w):
            nn[i, [0, w[0]]] = nn[i, [w[0], 0]]
        else:
            nn[i] = np.r_[i, nn[i, :-1]]

    rec = (nn[nn] == ar[:, None, None]).any(2)
    h = min(int(round(k1 / 2)), kk)
    nnh = nn[:, :h + 1]
    rech = (nnh[nnh] == ar[:, None, None]).any(2)
    Rfull = [nn[i][rec[i]] for i in range(N)]
    Rhalf = [set(nnh[i][rech[i]].tolist()) for i in range(N)]
    rows, cols, vals = [], [], []
    for i in range(N):
        R = Rfull[i]
        Rset = set(R.tolist())
        add = set()
        for c in R.tolist():
            rc = Rhalf[c]
            if rc and sum(1 for x in rc if x in Rset) > (2.0 / 3.0) * len(rc):
                add |= rc
        idx = np.fromiter(Rset | add, dtype=np.int64)
        w = np.exp(-D[i, idx])
        rows.append(np.full(len(idx), i, np.int64)); cols.append(idx); vals.append((w / w.sum()).astype(np.float32))
    V = sparse.csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(N, N))
    if k2 > 1:                                            # local query expansion
        k2e = min(k2, kk + 1)
        A = sparse.csr_matrix((np.full(N * k2e, 1.0 / k2e, np.float32), (np.repeat(ar, k2e), nn[:, :k2e].ravel())), shape=(N, N))
        V = (A @ V).tocsr()
    Vc = V.tocsc()
    rp, ri, rd = V.indptr, V.indices, V.data
    cp, ci, cd = Vc.indptr, Vc.indices, Vc.data

    J = np.empty((nq, ng), np.float32)                    # jaccard distance, queries x gallery
    for i in range(nq):
        cs, vs = ri[rp[i]:rp[i + 1]], rd[rp[i]:rp[i + 1]]
        st, en = cp[cs].astype(np.int64), cp[cs + 1].astype(np.int64)
        cnt = en - st
        tot = int(cnt.sum())
        gather = np.arange(tot) + np.repeat(st - (np.cumsum(cnt) - cnt), cnt)
        tm = np.bincount(ci[gather], weights=np.minimum(cd[gather], np.repeat(vs, cnt)), minlength=N)
        J[i] = (1.0 - tm / (2.0 - tm))[nq:]
    S_rerank = 1.0 - ((1.0 - lam) * J + lam * D[:nq, nq:])
    return S_rerank.astype(np.float32), S_cos

## 4. Viewpoint compatibility buckets

In [ ]:
BK_SAME, BK_ADJ, BK_OPP, BK_MISSING, BK_OTHER = 0, 1, 2, 3, 4
_ADJ, _OPP = set(), set()
for a, bs in {"left": ["frontleft", "backleft"], "right": ["frontright", "backright"],
              "front": ["frontleft", "frontright"], "back": ["backleft", "backright"]}.items():
    for b in bs:
        _ADJ |= {(a, b), (b, a)}
for a, b in [("left", "right"), ("front", "back"), ("up", "down")]:
    _OPP |= {(a, b), (b, a)}

def vp_bucket(vq, vg):
    if vq == "unk" or vg == "unk": return BK_MISSING
    if vq == vg:                   return BK_SAME
    if (vq, vg) in _ADJ:           return BK_ADJ
    if (vq, vg) in _OPP:           return BK_OPP
    return BK_OTHER

VP_VOCAB = sorted(set(tr.viewpoint) | set(te.viewpoint) | {"unk"})
VP_IDX = {v: i for i, v in enumerate(VP_VOCAB)}
VP_TABLE = np.array([[vp_bucket(a, b) for b in VP_VOCAB] for a in VP_VOCAB], dtype=np.int8)
vp_id_tr = tr.viewpoint.map(VP_IDX).values.astype(np.int16)
vp_id_te = te.viewpoint.map(VP_IDX).values.astype(np.int16)
print("viewpoints:", VP_VOCAB)

## 5. Test candidate pool (re-rank test vs. full train gallery, per species)

In [ ]:
def build_test_pool(sp):
    P_tr, P_te = EMB[sp]; sub = SUB[sp]
    t0 = time.time()
    Sr, Sc = k_reciprocal_rerank(P_te, P_tr)
    assert P_tr.shape[0] >= CAND_K, f"gallery has only {P_tr.shape[0]} images; lower CAND_K"
    part = np.argpartition(-Sr, CAND_K - 1, axis=1)[:, :CAND_K]
    cand = np.take_along_axis(part, np.argsort(-np.take_along_axis(Sr, part, 1), axis=1), 1)   # sorted by S_rerank desc
    vq, vg = vp_id_te[TE_SUB[sp]], vp_id_tr[sub]
    print(f"  {sp}: queries={len(P_te)} gallery={len(P_tr)} [{time.time()-t0:.0f}s]")
    return dict(Sr=np.take_along_axis(Sr, cand, 1), Sc=np.take_along_axis(Sc, cand, 1),
                bk=VP_TABLE[vq[:, None], vg[cand]], gc=code_tr[sub][cand].astype(np.int32),
                cand=sub[cand].astype(np.int32), q_rows=TE_SUB[sp])

POOL_TE = {sp: build_test_pool(sp) for sp in SPECIES}

## 6. SIFT + RANSAC verification of the top-100 candidates
Results are cached in `cache/sift_test_top100.parquet`. The cache is reused only when its signature, query ids and gallery ids all match, so a stale cache is never silently used.

In [ ]:
# ---------- SIFT source (.npz file, or an unzipped folder of <id>_xy.npy / <id>_desc.npy) ----------
def locate_sift(name):
    roots = [os.environ.get("SIFT_DIR"), PARENT_DIR, os.getcwd(), DATA_DIR]
    for r in roots:
        if not r: continue
        for nm in (name, name[:-4] if name.endswith(".npz") else name):
            p = os.path.join(r, nm)
            if os.path.exists(p):
                return p
    raise FileNotFoundError(f"{name} not found in {roots} - set SIFT_DIR to the folder that contains it.")

class SiftSource:
    def __init__(self, path):
        self.zs, self.dir, self.map = [], None, None
        if os.path.isfile(path):
            self.zs = [np.load(path)]
        else:
            npzs = sorted(glob.glob(os.path.join(path, "**", "*.npz"), recursive=True))
            if npzs: self.zs = [np.load(p) for p in npzs]
            else:    self.dir = path
        print("SIFT source:", path, "| mode:", "npz" if self.zs else "folder of .npy")

    def get(self, key):
        for z in self.zs:
            try: return z[key]
            except KeyError: continue
        if self.dir is not None:
            p = os.path.join(self.dir, key + ".npy")
            if os.path.exists(p): return np.load(p)
            if self.map is None:
                self.map = {os.path.splitext(os.path.basename(q))[0]: q for q in glob.glob(os.path.join(self.dir, "**", "*.npy"), recursive=True)}
            return np.load(self.map[key])
        raise KeyError(key)

def load_sift_for(ids_needed):
    """Load SIFT arrays only for the image ids that are actually needed."""
    srcs = [(SiftSource(locate_sift("sift_features_train.npz")), set(tr.id.values)),
            (SiftSource(locate_sift("sift_features_test.npz")),  set(te.id.values))]
    out, t0 = {}, time.time()
    for n, i in enumerate(ids_needed):
        for src, idset in srcs:
            if i in idset:
                out[i] = (src.get(f"{i}_xy").reshape(-1, 2).astype(np.float32), src.get(f"{i}_desc").reshape(-1, 128))
                break
        if (n + 1) % 2000 == 0: print(f"  loaded {n+1}/{len(ids_needed)} SIFT arrays [{time.time()-t0:.0f}s]")
    print(f"SIFT loaded: {len(out)} images [{time.time()-t0:.0f}s]")
    return out

# ---------- pair matching (identical logic to the tuning notebook) ----------
_tls = threading.local()
def _bf():
    if not hasattr(_tls, "bf"):
        _tls.bf = cv2.BFMatcher(cv2.NORM_L2)
    return _tls.bf

def _pair_stats(bf, xq, dq32, xg, dg):
    """(ratio_matches, ransac_inliers) for one image pair."""
    if len(xq) < 4 or len(xg) < 4:
        return 0, 0
    dg32 = dg.astype(np.float32)
    knn = bf.knnMatch(dq32, dg32, k=2)
    good = [m[0] for m in knn if len(m) == 2 and m[0].distance < SIFT_RATIO * m[1].distance]      # Lowe ratio test
    if not good:
        return 0, 0
    rev = bf.match(dg32, dq32)                                                                      # mutual NN check
    rt = np.fromiter((r.trainIdx for r in rev), dtype=np.int64, count=len(rev))
    good = [m for m in good if rt[m.trainIdx] == m.queryIdx]
    if len(good) < 4:
        return len(good), 0
    pq = np.float32([xq[m.queryIdx] for m in good]); pg = np.float32([xg[m.trainIdx] for m in good])
    cv2.setRNGSeed(0)
    _, mask = cv2.estimateAffinePartial2D(pq, pg, method=cv2.RANSAC, ransacReprojThreshold=RANSAC_THRESH, maxIters=RANSAC_ITERS)
    return len(good), (0 if mask is None else int(mask.sum()))

def run_sift(SIFT, q_ids, g_mat, label, chunk_q=20):
    """q_ids (nq,), g_mat (nq, M) gallery ids -> int32 (nq*M, 4): n_kp_q, n_kp_g, ratio_matches, ransac_inliers."""
    cv2.setNumThreads(1)
    def work(items):
        bf, out = _bf(), []
        for qid, gids in items:
            xq, dq = SIFT[qid]; dq32 = dq.astype(np.float32)
            for gid in gids:
                xg, dg = SIFT[gid]
                nm, ni = _pair_stats(bf, xq, dq32, xg, dg)
                out.append((len(xq), len(xg), nm, ni))
        return np.array(out, dtype=np.int32).reshape(-1, 4)
    tasks = [[(q_ids[i], g_mat[i].tolist()) for i in range(a, min(a + chunk_q, len(q_ids)))] for a in range(0, len(q_ids), chunk_q)]
    workers = max(1, os.cpu_count() or 1)
    print(f"[{label}] {len(q_ids)} queries x {g_mat.shape[1]} gallery = {len(q_ids)*g_mat.shape[1]:,} pairs on {workers} thread(s)")
    t0, res = time.time(), []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for n, r in enumerate(ex.map(work, tasks)):
            res.append(r)
            if (n + 1) % max(1, len(tasks) // 20) == 0:
                print(f"  {n+1}/{len(tasks)} chunks [{time.time()-t0:.0f}s]")
    print(f"[{label}] done [{time.time()-t0:.0f}s]")
    return np.concatenate(res)

# ---------- cache handling (same file names / signature as the tuning notebook, so its test cache is reused) ----------
M_SIFT = min(TOP_M_SIFT, CAND_K)
os.makedirs(CACHE_DIR, exist_ok=True)
META_PATH = os.path.join(CACHE_DIR, "sift_meta.json")
SIG = dict(M=M_SIFT, ratio=SIFT_RATIO, ransac=RANSAC_THRESH, iters=RANSAC_ITERS, k1=RERANK_K1, k2=RERANK_K2, lam=RERANK_LAM,
           center=EMB_CFG["center"], pca=EMB_CFG["pca"], cand_k=CAND_K, n_train=len(tr), n_test=len(te))

def make_pairs(pool, sp):
    g = pool["cand"][:, :M_SIFT]
    return pd.DataFrame(dict(fold=-1, query_id=np.repeat(te.id.values[pool["q_rows"]], M_SIFT),
                             gallery_id=tr.id.values[g].ravel(), gallery_identity=tr.identity.values[g].ravel(), species=sp))

def _save_tbl(df, path):
    try:
        df.to_parquet(path, index=False)
    except Exception as e:
        path = path.replace(".parquet", ".csv.gz"); df.to_csv(path, index=False)
        print("  (parquet unavailable: %s) saved %s" % (type(e).__name__, path))

def _load_tbl(path):
    if os.path.exists(path): return pd.read_parquet(path)
    if os.path.exists(path.replace(".parquet", ".csv.gz")): return pd.read_csv(path.replace(".parquet", ".csv.gz"))
    return None

def get_sift_table(pairs, kind="test"):
    path = os.path.join(CACHE_DIR, f"sift_{kind}_top{TOP_M_SIFT}.parquet")
    meta = json.load(open(META_PATH)) if os.path.exists(META_PATH) else {}
    cached = _load_tbl(path)
    if (cached is not None and meta.get(kind) == SIG and len(cached) == len(pairs)
            and (cached.query_id.values == pairs.query_id.values).all() and (cached.gallery_id.values == pairs.gallery_id.values).all()):
        print(f"[{kind}] loaded cache {path} ({len(cached):,} pairs)")
        return cached
    needed = list(dict.fromkeys(list(pairs.query_id.values[::M_SIFT]) + list(pairs.gallery_id.unique())))
    SIFT = load_sift_for(needed)
    st = run_sift(SIFT, pairs.query_id.values[::M_SIFT], pairs.gallery_id.values.reshape(-1, M_SIFT), kind)
    out = pairs.copy()
    out["n_kp_q"], out["n_kp_g"], out["ratio_matches"], out["ransac_inliers"] = st.T
    out["inlier_ratio"] = out.ransac_inliers / np.maximum(out.ratio_matches, 1)
    _save_tbl(out, path)
    meta[kind] = SIG
    json.dump(meta, open(META_PATH, "w"), indent=1)
    return out

pairs_te = pd.concat([make_pairs(POOL_TE[sp], sp) for sp in SPECIES], ignore_index=True)
sift_te = get_sift_table(pairs_te, "test")

off = 0                                                     # attach RANSAC inlier counts to each pool
for sp in SPECIES:
    P = POOL_TE[sp]; n = len(P["cand"])
    P["inl"] = np.zeros((n, CAND_K), np.float32)
    P["inl"][:, :M_SIFT] = sift_te.ransac_inliers.values[off:off + n * M_SIFT].reshape(n, M_SIFT)
    off += n * M_SIFT
assert off == len(sift_te)
print("pairs with 0 inliers: %.1f%%" % (100 * (sift_te.ransac_inliers.values == 0).mean()))

## 7. Blend scores, aggregate to identities, insert `new_individual`

In [ ]:
NEG = -1e6

def agg_topk(sc, gc, agg="max", topn=5):
    """sc, gc (nq, K): candidate-image scores / identity codes -> top identities (codes, scores), best first.
    Identity score = mean of its best k image scores among the candidates (k=1 'max', 2 'top2', ...)."""
    nq, K = sc.shape
    k = 1 if agg == "max" else int(agg[3:])
    o1 = np.argsort(-sc, axis=1)
    gc1 = np.take_along_axis(gc, o1, 1)
    o2 = np.argsort(gc1, axis=1, kind="stable")
    order = np.take_along_axis(o1, o2, 1)
    A, V = np.take_along_axis(gc, order, 1), np.take_along_axis(sc, order, 1)
    acc, cnt = V.astype(np.float32).copy(), np.ones((nq, K), np.float32)
    for j in range(1, k):
        same = A[:, j:] == A[:, :K - j]
        acc[:, :K - j] += np.where(same, V[:, j:], 0)
        cnt[:, :K - j] += same
    first = np.ones((nq, K), bool); first[:, 1:] = A[:, 1:] != A[:, :-1]
    gs = np.where(first, acc / cnt, NEG).astype(np.float32)
    kk = min(topn, K)
    part = np.argpartition(-gs, kk - 1, axis=1)[:, :kk]
    sel = np.take_along_axis(part, np.argsort(-np.take_along_axis(gs, part, 1), axis=1), 1)
    ids, scs = np.take_along_axis(A, sel, 1), np.take_along_axis(gs, sel, 1)
    return np.where(scs > NEG / 2, ids, -2), scs

def blend(P, p):
    btab = np.array([p["b_same"], p["b_adj"], p["b_opp"], 0.0, 0.0], np.float32)   # same, adjacent, opposite, missing, other
    lsift = np.log1p(np.maximum(P["inl"] - p["floor"], 0))
    return p["w_rerank"] * P["Sr"] + (1 - p["w_rerank"]) * P["Sc"] + p["w_sift"] * lsift + btab[P["bk"]]

def new_position(top_sc, tau):
    return np.minimum((top_sc >= tau).sum(1), 4)

def apply_new(top_names, top_sc, tau):
    pos, out = new_position(top_sc, tau), []
    for names, p in zip(top_names, pos):
        lst = list(names)
        lst.insert(int(p), NEW)
        out.append(lst[:5])
    return out

preds = pd.Series(index=te.index, dtype=object)
TOP1 = {}
for sp in SPECIES:                                           # species never mixed
    P = POOL_TE[sp]
    top, sc = agg_topk(blend(P, BEST_PARAMS), P["gc"], BEST_PARAMS["agg"])
    TOP1[sp] = sc[:, 0]
    names = np.where(top >= 0, CLASSES[sp][np.maximum(top, 0)], "")
    lists = [[n for n in l if n] for l in apply_new(names, sc, BEST_TAUS[sp])]
    preds[te.index[P["q_rows"]]] = [" ".join(l) for l in lists]
sub = pd.DataFrame({"id": te.id.values, "predictions": preds.values})
print(sub.head())

## 8. Sanity checks & write `submission.csv`

In [ ]:
ss = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert len(sub) == len(ss) == len(te), "row count"
assert set(sub.id) == set(ss.id) and sub.id.is_unique, "ids differ from sample_submission"
assert not sub.predictions.isna().any(), "NaN predictions"

ident_species = tr.drop_duplicates("identity").set_index("identity").species.to_dict()
test_species = te.set_index("id").species.to_dict()
new_rank = Counter()
for i, p in zip(sub.id, sub.predictions):
    labs = p.split()
    assert 1 <= len(labs) <= 5 and len(set(labs)) == len(labs), f"bad label count/dupes in {i}"
    for l in labs:
        assert l == NEW or (l in ident_species and ident_species[l] == test_species[i]), f"invalid label {l} for {i}"
    new_rank[labs.index(NEW) + 1 if NEW in labs else 0] += 1

print("ALL SANITY CHECKS PASSED")
print("new_individual rank distribution (0 = absent):", dict(sorted(new_rank.items())))
print(f"predicted new@1 rate: {new_rank[1] / len(sub):.1%}   (reference run: 39.8%)")
print(sub.sample(8, random_state=0).to_string(index=False))

sub.to_csv(OUT_PATH, index=False)
print("saved", os.path.abspath(OUT_PATH), "| rows:", len(sub))